# Six Degrees of MoMA: 03 GNNs and the curator ablation

This notebook trains encoder–decoder link predictors on rolling temporal snapshots and evaluates
them with the same protocol as the baselines.

- **Heterogeneous** (`rgcn`, `heteroconv`, `hgt`): artist, exhibition and curator nodes.
  Test exhibitions enter the graph with their metadata and curator edges only.
- **Homogeneous** (`sage`, `gat`): the artist co-exhibition graph. Exhibitions are embedded from
  metadata with an MLP.
- **Feature-only** (`mlp`): no graph at all, only artist and exhibition features. It shows how much
  the graph adds, and it is the learned cold-start baseline.
- **Curator modes**: `full`; `none` (curator node type dropped); `shuffled` (curator→exhibition
  assignments permuted within each decade, keeping all degrees).

Training runs on the GPU when one is available (Colab: Runtime → Change runtime type → GPU).
Outputs go to `results/gnn/`. Run `02_baselines.ipynb` first to include baselines in the final
comparison.

In [1]:
# Installs anything missing. Colab already has torch, pandas, scikit-learn and matplotlib.
import importlib.util, subprocess, sys

REQUIRED = {"torch": "torch", "torch_geometric": "torch_geometric", "pyarrow": "pyarrow",
            "sentence_transformers": "sentence-transformers", "scipy": "scipy",
            "sklearn": "scikit-learn", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [2]:
from pathlib import Path

# On Colab, set USE_DRIVE = True to keep data and results across sessions.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/moma-project")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
for d in [RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    # Raw ExhibitionRole values mapped onto node roles; everything else is "other".
    # Institutions listed with a curator role are treated as "other" (organizers).
    "roles": {"artist": ["Artist"], "curator": ["Curator"]},
    "features": {"title_encoder": "minilm",   # minilm | tfidf
                 "title_dim": 32, "top_nationalities": 12,
                 "collection": True},     # MoMA Collection acquisition counts per artist
    "split": {
        "cutoffs": [1950, 1960, 1970, 1980],
        "val_years": 5,       # val targets:  [T - val_years, T)
        "test_years": 10,     # test targets: [T, T + test_years)
        "train_window": 5,    # rolling training targets: [t, t + train_window)
        "train_start": 1932,  # earliest pseudo-cutoff for training snapshots
    },
    "eval": {"ks": [10, 20, 50], "primary_metric": "map@20"},
    "baselines": {
        "recency_tau": 10.0,
        "node2vec": {"dim": 64, "walk_length": 20, "walks_per_node": 10, "window": 4,
                     "p": 1.0, "q": 1.0, "epochs": 2, "num_neg": 5},
    },
    "model": {
        "name": "rgcn",           # rgcn | heteroconv | hgt | sage | gat | mlp
        "curator_mode": "full",   # full | none | shuffled
        "hidden": 64, "layers": 3, "dropout": 0.2, "lr": 0.005, "weight_decay": 1e-4,
        "epochs": 300, "eval_every": 5, "patience": 10,   # patience in evaluation rounds
        "loss": "bpr",            # bpr | bce
        "num_neg": 20,
        "curator_ids": True,      # learned embedding per curator (heterogeneous models)
        "max_group": None,        # subsample group shows above this size (homogeneous graph)
    },
    "seeds": [0, 1, 2, 3, 4],
}

## Core: data loading and cleaning

The CSV has one row per (exhibition, constituent, role) and is cp1252-encoded. MoMA's
collection data adds one row per acquired artwork; only the artist and acquisition date are kept.

- **Exhibitions** are keyed on (number, title, begin date). ExhibitionID is missing for about 430 rows, and
  the number "No#" covers about 50 different exhibitions. `exh_id` is the chronological index.
- **Constituents** without a ConstituentID are keyed by display name.
- **Roles**: `artist`, `curator` (an individual with `ExhibitionRole == "Curator"`) or `other`.

In [3]:
import copy, json, re, time, urllib.request
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://raw.githubusercontent.com/MuseumofModernArt/exhibitions/master/"
EXHIBITIONS_FILE = "MoMAExhibitions1929to1989.csv"
# MoMA's collection data (73 MB, stored with Git LFS, hence the media URL).
COLLECTION_URL = "https://media.githubusercontent.com/media/MuseumofModernArt/collection/main/"
COLLECTION_FILE = "Artworks.csv"
TABLES = ["exhibitions", "constituents", "participation", "acquisitions"]


def download(raw_dir, force=False):
    for url, name in [(BASE_URL, EXHIBITIONS_FILE), (COLLECTION_URL, COLLECTION_FILE)]:
        dest = raw_dir / name
        if force or not dest.exists():
            print(f"downloading {name}")
            urllib.request.urlretrieve(url + name, dest)


def load_raw(raw_dir):
    return pd.read_csv(raw_dir / EXHIBITIONS_FILE, encoding="cp1252", dtype=str)


def build_exhibitions(raw):
    ex = raw.drop_duplicates("exh_key")[[
        "exh_key", "ExhibitionID", "ExhibitionNumber", "ExhibitionTitle",
        "ExhibitionBeginDate", "ExhibitionEndDate", "ExhibitionSortOrder", "ExhibitionURL",
    ]].copy()
    ex["begin"] = pd.to_datetime(ex["ExhibitionBeginDate"], format="%m/%d/%Y")
    ex["end"] = pd.to_datetime(ex["ExhibitionEndDate"], format="%m/%d/%Y")
    ex["sort_order"] = pd.to_numeric(ex["ExhibitionSortOrder"], errors="coerce")
    ex = ex.sort_values(["begin", "sort_order", "exh_key"]).reset_index(drop=True)
    ex["exh_id"] = np.arange(len(ex))
    ex["year"] = ex["begin"].dt.year + (ex["begin"].dt.day_of_year - 1) / 365.25
    duration = (ex["end"] - ex["begin"]).dt.days
    ex["duration_days"] = duration.where(duration >= 0)
    return ex.rename(columns={
        "ExhibitionID": "source_id", "ExhibitionNumber": "number",
        "ExhibitionTitle": "title", "ExhibitionURL": "url",
    })[["exh_id", "exh_key", "source_id", "number", "title", "begin", "end", "year",
        "duration_days", "url"]]


def build_constituents(rows):
    def first_valid(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else None

    cols = {
        "DisplayName": "name", "ConstituentType": "type", "Nationality": "nationality",
        "Gender": "gender", "ConstituentBeginDate": "birth_year",
        "ConstituentEndDate": "death_year", "WikidataID": "wikidata", "ULANID": "ulan",
    }
    con = rows.groupby("cid")[list(cols)].agg(first_valid).rename(columns=cols).reset_index()
    con["gender"] = con["gender"].str.lower().fillna("unknown")
    nat = con["nationality"]
    con["nationality"] = nat.where(nat != "Nationality unknown").fillna("unknown")
    con["type"] = con["type"].fillna("unknown")
    for col in ["birth_year", "death_year"]:
        year = pd.to_numeric(con[col], errors="coerce")
        con[col] = year.where(year > 0)
    return con


def build_participation(rows, roles):
    raw_role = rows["ExhibitionRole"]
    role = pd.Series("other", index=rows.index)
    role[raw_role.isin(roles["artist"])] = "artist"
    is_curator = raw_role.isin(roles["curator"]) & (rows["ConstituentType"] != "Institution")
    role[is_curator] = "curator"
    part = pd.DataFrame({
        "exh_id": rows["exh_id"].values,
        "cid": rows["cid"].values,
        "role": role.values,
        "raw_role": raw_role.values,
        "press_role": rows["ExhibitionRoleinPressRelease"].values,
    })
    return part.drop_duplicates(["exh_id", "cid", "role"]).reset_index(drop=True)


def clean(raw, roles):
    raw = raw.copy()
    raw["exh_key"] = (raw["ExhibitionNumber"].fillna("") + "|" + raw["ExhibitionTitle"].fillna("")
                      + "|" + raw["ExhibitionBeginDate"].fillna(""))
    exhibitions = build_exhibitions(raw)
    raw["exh_id"] = raw["exh_key"].map(exhibitions.set_index("exh_key")["exh_id"])

    # Rows with no role and no constituent only record that an exhibition happened.
    rows = raw[raw["ExhibitionRole"].notna() & raw["DisplayName"].notna()].copy()
    rows["cid"] = ("moma:" + rows["ConstituentID"]).fillna("name:" + rows["DisplayName"].str.strip())
    constituents = build_constituents(rows)
    participation = build_participation(rows, roles)

    has_curator = participation.loc[participation["role"] == "curator", "exh_id"].unique()
    exhibitions["has_curator"] = exhibitions["exh_id"].isin(has_curator)
    return {"exhibitions": exhibitions, "constituents": constituents,
            "participation": participation}


def load_acquisitions(raw_dir):
    """One row per (artwork, artist) acquired by MoMA, with the acquisition date. Artists are
    keyed like exhibition constituents ("moma:<ConstituentID>"), which is the same id space."""
    works = pd.read_csv(raw_dir / COLLECTION_FILE, dtype=str,
                        usecols=["ObjectID", "ConstituentID", "DateAcquired"])
    works["acquired"] = pd.to_datetime(works["DateAcquired"], errors="coerce")
    works = works.dropna(subset=["ConstituentID", "acquired"])
    works = works.assign(cid=works["ConstituentID"].str.split(",")).explode("cid")
    works["cid"] = "moma:" + works["cid"].str.strip()
    return works[["cid", "ObjectID", "acquired"]].rename(columns={"ObjectID": "object_id"}) \
        .drop_duplicates().reset_index(drop=True)


def load_tables(force=False):
    """Cleaned tables from data/processed, rebuilding them from the raw CSV if needed."""
    if not force and all((PROCESSED_DIR / f"{n}.parquet").exists() for n in TABLES):
        return {n: pd.read_parquet(PROCESSED_DIR / f"{n}.parquet") for n in TABLES}
    download(RAW_DIR)
    tables = clean(load_raw(RAW_DIR), CFG["roles"])
    tables["acquisitions"] = load_acquisitions(RAW_DIR)
    for name, df in tables.items():
        df.to_parquet(PROCESSED_DIR / f"{name}.parquet", index=False)
    return tables

### Integer-indexed dataset

Artists and curators have separate node index spaces. A person who both exhibited and curated
appears once in each; both keep `cid` so they can be merged back into one person when needed.

`with_shuffled_curators` is the shuffle ablation. It permutes curator→exhibition assignments
within each decade, so every curator keeps their number of exhibitions and every exhibition
keeps its number of curators.

In [4]:
@dataclass
class MomaData:
    exhibitions: pd.DataFrame    # indexed by exh_id (chronological)
    artists: pd.DataFrame        # indexed by artist node id
    curators: pd.DataFrame       # indexed by curator node id
    artist_edges: pd.DataFrame   # columns exh_id, artist
    curator_edges: pd.DataFrame  # columns exh_id, curator
    acquisitions: pd.DataFrame   # columns artist, year: collection artworks by acquisition date

    @classmethod
    def from_tables(cls, tables):
        ex = tables["exhibitions"].set_index("exh_id", drop=False).sort_index()
        con = tables["constituents"].set_index("cid")
        part = tables["participation"]

        def role_nodes(role):
            edges = part[part["role"] == role][["exh_id", "cid"]]
            cids = np.sort(edges["cid"].unique())
            nodes = con.loc[cids].reset_index()
            node_of = pd.Series(np.arange(len(cids)), index=cids)
            edges = pd.DataFrame({"exh_id": edges["exh_id"].to_numpy(),
                                  role: node_of.loc[edges["cid"]].to_numpy()})
            return nodes, edges.sort_values(["exh_id", role]).reset_index(drop=True)

        artists, artist_edges = role_nodes("artist")
        curators, curator_edges = role_nodes("curator")

        acq = tables["acquisitions"]
        artist_of = pd.Series(artists.index, index=artists["cid"])
        acq = acq[acq["cid"].isin(artist_of.index)]
        date = acq["acquired"].dt
        acquisitions = pd.DataFrame({
            "artist": artist_of.loc[acq["cid"]].to_numpy(),
            "year": (date.year + (date.day_of_year - 1) / 365.25).to_numpy(),
        }).sort_values("year", ignore_index=True)
        return cls(ex, artists, curators, artist_edges, curator_edges, acquisitions)

    @property
    def num_artists(self):
        return len(self.artists)

    @property
    def num_curators(self):
        return len(self.curators)

    @property
    def num_exhibitions(self):
        return len(self.exhibitions)

    def with_shuffled_curators(self, seed):
        rng = np.random.default_rng(seed)
        edges = self.curator_edges.copy()
        decade = (self.exhibitions.loc[edges["exh_id"], "begin"].dt.year // 10).to_numpy()
        exh = edges["exh_id"].to_numpy().copy()
        for d in np.unique(decade):
            idx = np.flatnonzero(decade == d)
            exh[idx] = exh[rng.permutation(idx)]
        edges["exh_id"] = exh
        # A permutation can rarely pair a curator with the same exhibition twice.
        edges = edges.drop_duplicates().sort_values(["exh_id", "curator"]).reset_index(drop=True)
        return replace(self, curator_edges=edges)

## Core: temporal snapshots

A snapshot fixes a history cutoff and a set of target exhibitions after it. A model may see:

- every artist and curator edge of exhibitions that began before `history_end`;
- the curator edges and metadata of the target exhibitions.

It must rank the `candidates` (artists with at least one exhibition in the history) for each
target. The targets' artist edges are labels only.

For a cutoff year T:

| split | history | targets |
|---|---|---|
| train (rolling) | before t, for each t < T−5 | [t, min(t+5, T−5)) |
| validation | before T−5 | [T−5, T) |
| test | before T | [T, T+10) |

Rolling training snapshots have the same form as the test task, and the model never sees edges
from after its own targets.

`cold_start_snapshot` keeps a snapshot's targets but swaps the candidates for artists who make
their MoMA debut in the window. It is evaluated separately, because no history-based score can
rank those artists.

In [5]:
@dataclass
class Snapshot:
    history_end: pd.Timestamp
    targets: np.ndarray         # exh_ids, chronological
    hist_artist: np.ndarray     # [2, E] (exh_id, artist) for history exhibitions
    hist_curator: np.ndarray    # [2, E] (exh_id, curator) for history exhibitions
    target_curator: np.ndarray  # [2, E] (exh_id, curator) for target exhibitions
    candidates: np.ndarray      # artist ids with >=1 history exhibition, sorted
    positives: dict             # target exh_id -> all participating artist ids
    has_curator: np.ndarray     # per target: recorded curator in the original data

    @property
    def ref_year(self):
        ts = self.history_end
        return ts.year + (ts.day_of_year - 1) / 365.25

    @property
    def curator_edges(self):
        return np.concatenate([self.hist_curator, self.target_curator], axis=1)

    def label_matrix(self):
        """Boolean [num_targets, num_candidates]; True where the candidate exhibited."""
        labels = np.zeros((len(self.targets), len(self.candidates)), dtype=bool)
        for i, e in enumerate(self.targets):
            labels[i] = np.isin(self.candidates, self.positives[e], assume_unique=True)
        return labels


def _edges(df, col, mask):
    sub = df[mask]
    return np.stack([sub["exh_id"].to_numpy(), sub[col].to_numpy()]).astype(np.int64)


def make_snapshot(data, target_start, target_end, history_end=None):
    target_start, target_end = pd.Timestamp(target_start), pd.Timestamp(target_end)
    history_end = pd.Timestamp(history_end) if history_end is not None else target_start
    assert history_end <= target_start

    ex = data.exhibitions
    hist_ids = ex.index[ex["begin"] < history_end].to_numpy()
    targets = ex.index[(ex["begin"] >= target_start) & (ex["begin"] < target_end)].to_numpy().copy()

    ae, ce = data.artist_edges, data.curator_edges
    hist_artist = _edges(ae, "artist", ae["exh_id"].isin(hist_ids))
    hist_curator = _edges(ce, "curator", ce["exh_id"].isin(hist_ids))
    target_curator = _edges(ce, "curator", ce["exh_id"].isin(targets))

    target_artists = ae[ae["exh_id"].isin(targets)].groupby("exh_id")["artist"]
    positives = {e: np.array([], dtype=np.int64) for e in targets}
    positives.update({e: g.to_numpy() for e, g in target_artists})

    return Snapshot(
        history_end=history_end,
        targets=targets,
        hist_artist=hist_artist,
        hist_curator=hist_curator,
        target_curator=target_curator,
        candidates=np.unique(hist_artist[1]),
        positives=positives,
        has_curator=ex.loc[targets, "has_curator"].to_numpy(),
    )


def cold_start_snapshot(data, snap):
    """The same targets, ranked against the window's debuting artists instead of returning ones.

    Candidates are artists with no history who exhibit in at least one target. Knowing who debuts
    somewhere in the window is an oracle assumption; the task is to predict where."""
    appearing = np.concatenate([np.array([], dtype=np.int64), *snap.positives.values()])
    return replace(snap, candidates=np.setdiff1d(appearing, snap.candidates))


def year_start(year):
    return pd.Timestamp(year=int(year), month=1, day=1)


def eval_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    val = make_snapshot(data, year_start(val_start), year_start(cutoff))
    test = make_snapshot(data, year_start(cutoff), year_start(cutoff + split_cfg["test_years"]))
    return val, test


def train_snapshots(data, cutoff, split_cfg):
    val_start = cutoff - split_cfg["val_years"]
    snaps = []
    for t in range(split_cfg["train_start"], val_start):
        end = min(t + split_cfg["train_window"], val_start)
        snap = make_snapshot(data, year_start(t), year_start(end))
        if len(snap.targets) and len(snap.candidates):
            snaps.append(snap)
    return snaps

## Core: metrics and evaluation protocol

Every model is a scorer with `score(snapshot) -> [num_targets, num_candidates]`, so candidate
pools, tie-breaking and metrics are identical across baselines and GNNs. Ties are broken at
random, not by candidate order.

The **primary subset** is test exhibitions that have a recorded curator and at least one returning
artist. Exhibitions whose artists are all MoMA debuts are kept with `n_pos_in_pool = 0` for the
cold-start analysis.

In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score


def rank_order(scores, rng):
    """Indices sorted by descending score, ties broken uniformly at random."""
    return np.lexsort((rng.random(len(scores)), -scores))


def ranking_metrics(scores, labels, ks, rng):
    """Per-query metrics for one exhibition. `labels` is a boolean vector with >=1 True."""
    order = rank_order(scores, rng)
    rel = labels[order].astype(float)
    n_pos = rel.sum()
    hits = np.cumsum(rel)
    precision_at = hits / np.arange(1, len(rel) + 1)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))

    out = {}
    for k in ks:
        out[f"recall@{k}"] = hits[min(k, len(rel)) - 1] / n_pos
        out[f"map@{k}"] = (precision_at[:k] * rel[:k]).sum() / min(n_pos, k)
        ideal = discounts[:int(min(n_pos, k))].sum()
        out[f"ndcg@{k}"] = (discounts[:k] * rel[:k]).sum() / ideal
    if n_pos < len(rel):
        out["auc"] = roc_auc_score(labels, scores)
        out["ap"] = average_precision_score(labels, scores)
    else:
        out["auc"] = out["ap"] = np.nan

    ranks = np.empty(len(rel), dtype=np.int64)
    ranks[order] = np.arange(1, len(rel) + 1)
    return out, ranks[labels]


def evaluate(scorer, snap, ks, seed=0):
    """Returns (per-exhibition metrics, per-positive ranks)."""
    rng = np.random.default_rng(seed)
    scores = np.asarray(scorer.score(snap), dtype=np.float64)
    assert scores.shape == (len(snap.targets), len(snap.candidates))
    labels = snap.label_matrix()

    rows, rank_rows = [], []
    for i, e in enumerate(snap.targets):
        n_in_pool = int(labels[i].sum())
        row = {"exh_id": e, "has_curator": bool(snap.has_curator[i]),
               "n_pos": len(snap.positives[e]), "n_pos_in_pool": n_in_pool,
               "n_candidates": len(snap.candidates)}
        if n_in_pool:
            metrics, ranks = ranking_metrics(scores[i], labels[i], ks, rng)
            row.update(metrics)
            rank_rows.append(pd.DataFrame({
                "exh_id": e, "artist": snap.candidates[labels[i]], "rank": ranks,
                "n_candidates": len(snap.candidates)}))
        rows.append(row)
    per_exh = pd.DataFrame(rows)
    ranks = pd.concat(rank_rows, ignore_index=True) if rank_rows else pd.DataFrame()
    return per_exh, ranks


def metric_columns(per_exh):
    return [c for c in per_exh.columns if "@" in c or c in ("auc", "ap")]


def summarize(per_exh):
    """Mean metrics on the curated subset and on all exhibitions with >=1 returning artist."""
    scored = per_exh[per_exh["n_pos_in_pool"] > 0]
    cols = metric_columns(per_exh)
    return {name: {"n_exhibitions": len(subset), **subset[cols].mean().to_dict()}
            for name, subset in [("curated", scored[scored["has_curator"]]), ("all", scored)]}


def paired_bootstrap(a, b, n_boot=1000, seed=0, ci=0.95):
    """CI for mean(a - b), resampling the paired units (test exhibitions)."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    diff = diff[~np.isnan(diff)]
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    lo, hi = np.quantile(means, [(1 - ci) / 2, 1 - (1 - ci) / 2])
    return {"mean_diff": diff.mean(), "ci_low": lo, "ci_high": hi, "n": len(diff),
            "p_le_0": float((means <= 0).mean())}


RESULT_FILE = re.compile(r"(?P<tag>.+)_T(?P<cutoff>\d{4})(?:_s(?P<seed>\d+))?_"
                         r"(?P<kind>per_exh|ranks|cold_per_exh|cold_ranks)\.csv")


def result_files(tag, cutoff, kind="per_exh"):
    """Saved result files of one run tag: one per seed for GNNs, a single file for baselines.
    `kind` is per_exh or ranks, prefixed with cold_ for the cold-start evaluation."""
    files = []
    for d in ["gnn", "baselines"]:
        for f in sorted((RESULTS_DIR / d).glob(f"{tag}_T{cutoff}*.csv")):
            m = RESULT_FILE.fullmatch(f.name)
            if m and m["tag"] == tag and m["kind"] == kind:
                files.append(f)
    return files


def load_per_exh(tag, cutoff, kind="per_exh"):
    """Saved per-exhibition results (averaged over seeds) on the curated subset."""
    files = result_files(tag, cutoff, kind)
    if not files:
        return None, 0
    runs = pd.concat([pd.read_csv(f) for f in files])
    scored = runs[(runs["n_pos_in_pool"] > 0) & runs["has_curator"]]
    return scored.groupby("exh_id").mean(numeric_only=True), len(files)


def compare_runs(a, b, metric="map@20", cutoffs=None, n_boot=1000, kind="per_exh"):
    """Paired bootstrap of A - B over curated test exhibitions, per cutoff.

    Tags: baselines by name (e.g. "curator_collab"); GNNs as "{model}_{mode}" (e.g. "rgcn_none").
    """
    rows = []
    for cutoff in cutoffs or CFG["split"]["cutoffs"]:
        (ra, na), (rb, nb) = load_per_exh(a, cutoff, kind), load_per_exh(b, cutoff, kind)
        if ra is None or rb is None:
            continue
        joined = ra[[metric]].merge(rb[[metric]], left_index=True, right_index=True,
                                    suffixes=("_a", "_b"))
        res = paired_bootstrap(joined[f"{metric}_a"], joined[f"{metric}_b"], n_boot=n_boot)
        rows.append({"cutoff": cutoff, "seeds_a": na, "seeds_b": nb,
                     a: joined[f"{metric}_a"].mean(), b: joined[f"{metric}_b"].mean(), **res})
    return pd.DataFrame(rows)

## Load data

In [7]:
tables = load_tables()
data = MomaData.from_tables(tables)
print(f"{data.num_exhibitions} exhibitions, {data.num_artists} artists, "
      f"{data.num_curators} curators, {len(data.artist_edges)} artist edges, "
      f"{len(data.curator_edges)} curator edges")

1787 exhibitions, 11273 artists, 316 curators, 32537 artist edges, 1483 curator edges


In [8]:
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv, HGTConv, RGCNConv, SAGEConv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

/Users/shirleyzhang/Desktop/CS 224w/cs224w-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu


/Users/shirleyzhang/Desktop/CS 224w/cs224w-project/.venv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Node features

- **Static** features are demographics and exhibition metadata: year, duration, and a MiniLM
  title embedding reduced with PCA.
- **History** features are exhibition count and years since the first and last exhibition. They
  are recomputed per snapshot from its visible history only.
- **Collection** features (artists only) are the number of the artist's works MoMA had acquired
  before the snapshot's cutoff, and years since the first acquisition. Debuting artists have no
  exhibition history but often have collection history.
- Exhibition features deliberately exclude curator counts, so the `none` ablation cannot recover
  curator information through them.

In [9]:
YEAR_MIN, YEAR_SPAN = 1929, 60.0


def one_hot(values, categories):
    cat = pd.Categorical(values, categories=categories)
    out = np.zeros((len(values), len(categories)), dtype=np.float32)
    known = cat.codes >= 0
    out[np.flatnonzero(known), cat.codes[known]] = 1
    return out


def title_embeddings(titles, method, dim, seed=0):
    cache = PROCESSED_DIR / f"title_emb_{method}_{dim}.npy"
    if cache.exists():
        return np.load(cache)
    titles = [t if isinstance(t, str) else "" for t in titles]
    if method == "minilm":
        from sentence_transformers import SentenceTransformer
        from sklearn.decomposition import PCA
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        full = model.encode(titles, batch_size=128, show_progress_bar=False)
        emb = PCA(n_components=dim, random_state=seed).fit_transform(full)
    elif method == "tfidf":
        from sklearn.decomposition import TruncatedSVD
        from sklearn.feature_extraction.text import TfidfVectorizer
        tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2)).fit_transform(titles)
        emb = TruncatedSVD(n_components=dim, random_state=seed).fit_transform(tfidf)
    else:
        raise ValueError(method)
    emb = (emb / (emb.std(axis=0, keepdims=True) + 1e-8)).astype(np.float32)
    np.save(cache, emb)
    return emb


def exhibition_features(data, fcfg):
    ex = data.exhibitions
    duration = ex["duration_days"]
    log_dur = np.log1p(duration.fillna(duration.median())).to_numpy()
    base = np.stack([
        (ex["year"].to_numpy() - YEAR_MIN) / YEAR_SPAN,
        (log_dur - log_dur.mean()) / log_dur.std(),
        duration.isna().to_numpy().astype(float),
    ], axis=1).astype(np.float32)
    titles = title_embeddings(ex["title"].tolist(), fcfg["title_encoder"], fcfg["title_dim"])
    return np.concatenate([base, titles], axis=1)


def person_static_features(nodes, top_nationalities):
    nat = nodes["nationality"]
    top = nat[nat != "unknown"].value_counts().index[:top_nationalities].tolist()
    nat_bucket = np.where(nat.isin(top), nat, np.where(nat == "unknown", "unknown", "other"))
    birth = nodes["birth_year"]
    return np.concatenate([
        one_hot(nodes["gender"], ["male", "female", "unknown"]),
        one_hot(nat_bucket, top + ["other", "unknown"]),
        np.stack([
            ((birth.fillna(1900) - 1900) / 50.0).to_numpy(),
            birth.isna().to_numpy().astype(float),
            (nodes["type"] == "Institution").to_numpy().astype(float),
        ], axis=1),
    ], axis=1).astype(np.float32)


def history_features(edges, num_nodes, exh_years, ref_year):
    """[num_nodes, 4]: log count, years since first / last exhibition (decades), has-history."""
    years = exh_years[edges[0]]
    count = np.bincount(edges[1], minlength=num_nodes)
    first = np.full(num_nodes, np.inf)
    last = np.full(num_nodes, -np.inf)
    np.minimum.at(first, edges[1], years)
    np.maximum.at(last, edges[1], years)
    seen = count > 0
    return np.stack([
        np.log1p(count),
        np.where(seen, (ref_year - first) / 10.0, 0.0),
        np.where(seen, (ref_year - last) / 10.0, 0.0),
        seen.astype(float),
    ], axis=1).astype(np.float32)


def collection_features(acquisitions, num_nodes, ref_year):
    """[num_nodes, 3]: log count of works acquired before ref_year, has-any, years since first."""
    seen = acquisitions[acquisitions["year"] < ref_year]
    artist, year = seen["artist"].to_numpy(), seen["year"].to_numpy()
    count = np.bincount(artist, minlength=num_nodes)
    first = np.full(num_nodes, np.inf)
    np.minimum.at(first, artist, year)
    has = count > 0
    return np.stack([np.log1p(count), has.astype(float),
                     np.where(has, (ref_year - first) / 10.0, 0.0)], axis=1).astype(np.float32)


class FeatureStore:
    def __init__(self, data, cfg):
        fcfg = cfg["features"]
        self.data = data
        self.exh_years = data.exhibitions["year"].to_numpy()
        self.exhibition = exhibition_features(data, fcfg)
        self.artist_static = person_static_features(data.artists, fcfg["top_nationalities"])
        self.curator_static = person_static_features(data.curators, fcfg["top_nationalities"])
        self.use_collection = fcfg["collection"]

    def artist(self, snap):
        """Static, then collection (optional), then history features (always last)."""
        parts = [self.artist_static]
        if self.use_collection:
            parts.append(collection_features(self.data.acquisitions, self.data.num_artists,
                                             snap.ref_year))
        parts.append(history_features(snap.hist_artist, self.data.num_artists, self.exh_years,
                                      snap.ref_year))
        return np.concatenate(parts, axis=1)

    def curator(self, snap):
        hist = history_features(snap.hist_curator, self.data.num_curators, self.exh_years,
                                snap.ref_year)
        return np.concatenate([self.curator_static, hist], axis=1)


feats = FeatureStore(data, CFG)
print("exhibition features:", feats.exhibition.shape, " artist static:", feats.artist_static.shape)

exhibition features: (1787, 35)  artist static: (11273, 20)


## Graph construction

Every graph contains all nodes in the dataset; only the edges are snapshot-specific. Node ids
therefore stay stable, and nodes without visible history are isolated. `leak=True` exposes target
labels to message passing. It is used only for the sanity check below.

In [10]:
ARTIST, EXHIBITION, CURATOR = "artist", "exhibition", "curator"
PARTICIPATES = (ARTIST, "participates", EXHIBITION)
HAS_ARTIST = (EXHIBITION, "has_artist", ARTIST)
CURATES = (CURATOR, "curates", EXHIBITION)
CURATED_BY = (EXHIBITION, "curated_by", CURATOR)


def edge_types(include_curators):
    return [PARTICIPATES, HAS_ARTIST] + ([CURATES, CURATED_BY] if include_curators else [])


def _exhibition_x(snap, feats):
    rel_year = (feats.exh_years - snap.ref_year)[:, None] / 10.0
    return torch.from_numpy(np.concatenate([feats.exhibition, rel_year], axis=1).astype(np.float32))


def _artist_edges(snap, leak):
    edges = snap.hist_artist
    if leak:
        leaked = [np.stack([np.full(len(a), e), a]) for e, a in snap.positives.items()]
        edges = np.concatenate([edges] + leaked, axis=1)
    return torch.from_numpy(edges)


def build_hetero(snap, feats, include_curators=True, leak=False):
    g = HeteroData()
    g[ARTIST].x = torch.from_numpy(feats.artist(snap))
    g[EXHIBITION].x = _exhibition_x(snap, feats)
    exh_artist = _artist_edges(snap, leak)
    g[PARTICIPATES].edge_index = exh_artist.flip(0)
    g[HAS_ARTIST].edge_index = exh_artist
    if include_curators:
        g[CURATOR].x = torch.from_numpy(feats.curator(snap))
        exh_curator = torch.from_numpy(snap.curator_edges)
        g[CURATES].edge_index = exh_curator.flip(0)
        g[CURATED_BY].edge_index = exh_curator
    return g


def coexhibition_edges(hist_artist, max_group=None, rng=None):
    """Artist-artist edges (both directions) between artists who shared an exhibition.
    `max_group` optionally subsamples very large group shows, whose cliques dominate."""
    order = np.argsort(hist_artist[0], kind="stable")
    exh, artists = hist_artist[0][order], hist_artist[1][order]
    src, dst = [], []
    for group in np.split(artists, np.flatnonzero(np.diff(exh)) + 1):
        if max_group is not None and len(group) > max_group:
            group = rng.choice(group, max_group, replace=False)
        if len(group) < 2:
            continue
        a, b = np.meshgrid(group, group)
        mask = a != b
        src.append(a[mask])
        dst.append(b[mask])
    if not src:
        return np.zeros((2, 0), dtype=np.int64)
    return np.unique(np.stack([np.concatenate(src), np.concatenate(dst)]), axis=1).astype(np.int64)


def build_homogeneous(snap, feats, max_group=None, seed=0, leak=False):
    g = HeteroData()
    g[ARTIST].x = torch.from_numpy(feats.artist(snap))
    g[EXHIBITION].x = _exhibition_x(snap, feats)
    edges = coexhibition_edges(_artist_edges(snap, leak).numpy(), max_group,
                               np.random.default_rng(seed))
    g[ARTIST, "coexhibits", ARTIST].edge_index = torch.from_numpy(edges)
    return g

In [11]:
# Leakage check on the 1960 test graph: target exhibitions carry curator edges but no artists,
# and artist history counts include only history exhibitions.
_, test_1960 = eval_snapshots(data, 1960, CFG["split"])
g = build_hetero(test_1960, feats)
assert set(g[HAS_ARTIST].edge_index[0].tolist()).isdisjoint(test_1960.targets.tolist())
assert set(test_1960.target_curator[0]) <= set(g[CURATED_BY].edge_index[0].tolist())
counts = torch.expm1(g[ARTIST].x[:, -4]).round().long().numpy()
assert np.array_equal(counts, np.bincount(test_1960.hist_artist[1], minlength=data.num_artists))
if feats.use_collection:   # collection counts include only works acquired before 1960
    works = torch.expm1(g[ARTIST].x[:, -7]).round().long().numpy()
    assert np.array_equal(works, np.bincount(
        data.acquisitions.loc[data.acquisitions["year"] < 1960, "artist"],
        minlength=data.num_artists))

# Shuffle ablation preserves curator degrees (up to rare duplicate pairs) but moves most edges.
shuffled = data.with_shuffled_curators(seed=0)
before = data.curator_edges.groupby("curator").size()
after = shuffled.curator_edges.groupby("curator").size().reindex(before.index, fill_value=0)
assert (before - after).abs().sum() <= 0.02 * len(data.curator_edges)
assert len(data.curator_edges.merge(shuffled.curator_edges)) < 0.5 * len(data.curator_edges)
print("graph checks passed")
g

graph checks passed


HeteroData(
  artist={ x=[11273, 27] },
  exhibition={ x=[1787, 36] },
  curator={ x=[316, 15] },
  (artist, participates, exhibition)={ edge_index=[2, 16457] },
  (exhibition, has_artist, artist)={ edge_index=[2, 16457] },
  (curator, curates, exhibition)={ edge_index=[2, 736] },
  (exhibition, curated_by, curator)={ edge_index=[2, 736] }
)

## Models

Each encoder maps a snapshot graph to artist and exhibition embeddings. Node features are
attributes and history counts, not identities, so with `curator_ids` the heterogeneous models
also learn one embedding per curator. Curators recur across decades (there are only 316 of them),
which lets the model tell "Lieberman curated this" from "some curator with a similar profile
did". The two-tower decoder
scores pairs as score(a, e) = ⟨f(h_a), g(h_e)⟩ + b(h_a). The artist bias lets the model represent
popularity directly. Scoring every candidate for every target is then a single matrix product.

In [12]:
class TypeProjection(nn.Module):
    """Linear projection of each node type's features. Types in `num_ids` also get a learned
    embedding per node, so that recurring nodes (curators) can be recognized by identity."""

    def __init__(self, in_dims, hidden, num_ids=None):
        super().__init__()
        self.lins = nn.ModuleDict({t: nn.Linear(d, hidden) for t, d in in_dims.items()})
        self.ids = nn.ModuleDict({t: nn.Embedding(n, hidden) for t, n in (num_ids or {}).items()})
        for emb in self.ids.values():
            nn.init.zeros_(emb.weight)   # unseen nodes start from their features alone

    def forward(self, x_dict):
        return {t: self.lins[t](x) + (self.ids[t].weight if t in self.ids else 0)
                for t, x in x_dict.items()}


class RGCNEncoder(nn.Module):
    def __init__(self, in_dims, edge_types, hidden, layers, dropout, num_ids=None):
        super().__init__()
        self.node_types = list(in_dims)
        self.edge_types = list(edge_types)
        self.proj = TypeProjection(in_dims, hidden, num_ids)
        self.convs = nn.ModuleList(
            RGCNConv(hidden, hidden, num_relations=len(self.edge_types)) for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h_dict = self.proj(g.x_dict)
        offsets, start = {}, 0
        for t in self.node_types:
            offsets[t] = start
            start += h_dict[t].shape[0]
        x = torch.cat([h_dict[t] for t in self.node_types])

        edge_index, edge_type = [], []
        for rel, (src, name, dst) in enumerate(self.edge_types):
            ei = g[src, name, dst].edge_index
            edge_index.append(torch.stack([ei[0] + offsets[src], ei[1] + offsets[dst]]))
            edge_type.append(torch.full((ei.shape[1],), rel, dtype=torch.long, device=ei.device))
        edge_index, edge_type = torch.cat(edge_index, 1), torch.cat(edge_type)

        x = self.dropout(torch.relu(x))
        for conv in self.convs:
            x = x + self.dropout(torch.relu(conv(x, edge_index, edge_type)))
        out = {t: x[offsets[t]:offsets[t] + h_dict[t].shape[0]] for t in self.node_types}
        return out[ARTIST], out[EXHIBITION]


class HeteroSAGEEncoder(nn.Module):
    """HeteroConv with a separate GraphSAGE operator per relation."""

    def __init__(self, in_dims, edge_types, hidden, layers, dropout, num_ids=None):
        super().__init__()
        self.proj = TypeProjection(in_dims, hidden, num_ids)
        self.convs = nn.ModuleList(
            HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr="sum")
            for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h = {t: self.dropout(torch.relu(x)) for t, x in self.proj(g.x_dict).items()}
        for conv in self.convs:
            update = conv(h, g.edge_index_dict)
            h = {t: h[t] + self.dropout(torch.relu(update[t])) if t in update else h[t]
                 for t in h}
        return h[ARTIST], h[EXHIBITION]


class HGTEncoder(nn.Module):
    def __init__(self, in_dims, edge_types, hidden, layers, dropout, num_ids=None, heads=2):
        super().__init__()
        metadata = (list(in_dims), list(edge_types))
        self.proj = TypeProjection(in_dims, hidden, num_ids)
        self.convs = nn.ModuleList(HGTConv(hidden, hidden, metadata, heads=heads)
                                   for _ in range(layers))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        h = {t: self.dropout(torch.relu(x)) for t, x in self.proj(g.x_dict).items()}
        for conv in self.convs:
            update = conv(h, g.edge_index_dict)
            h = {t: h[t] + self.dropout(update[t]) if update.get(t) is not None else h[t]
                 for t in h}
        return h[ARTIST], h[EXHIBITION]


class HomogeneousEncoder(nn.Module):
    """GraphSAGE or GAT over the artist co-exhibition graph; MLP over exhibition metadata."""

    def __init__(self, in_dims, hidden, layers, dropout, conv="sage", heads=2):
        super().__init__()
        self.artist_in = nn.Linear(in_dims[ARTIST], hidden)
        if conv == "sage":
            make = lambda: SAGEConv(hidden, hidden)
        elif conv == "gat":
            make = lambda: GATConv(hidden, hidden // heads, heads=heads)
        else:
            raise ValueError(conv)
        self.convs = nn.ModuleList(make() for _ in range(layers))
        self.exhibition_mlp = nn.Sequential(
            nn.Linear(in_dims[EXHIBITION], hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden))
        self.dropout = nn.Dropout(dropout)

    def forward(self, g):
        edge_index = g[ARTIST, "coexhibits", ARTIST].edge_index
        x = self.dropout(torch.relu(self.artist_in(g[ARTIST].x)))
        for conv in self.convs:
            x = x + self.dropout(torch.relu(conv(x, edge_index)))
        return x, self.exhibition_mlp(g[EXHIBITION].x)


class FeatureEncoder(nn.Module):
    """No message passing: separate MLPs over artist and exhibition features. This is the
    graph-free baseline, and for debuting artists the learned version of the attribute
    baselines."""

    def __init__(self, in_dims, hidden, dropout):
        super().__init__()

        def mlp(d):
            return nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden))

        self.artist_mlp, self.exhibition_mlp = mlp(in_dims[ARTIST]), mlp(in_dims[EXHIBITION])

    def forward(self, g):
        return self.artist_mlp(g[ARTIST].x), self.exhibition_mlp(g[EXHIBITION].x)


HETERO_MODELS = {"rgcn": RGCNEncoder, "heteroconv": HeteroSAGEEncoder, "hgt": HGTEncoder}
HOMOGENEOUS_MODELS = {"sage", "gat"}


class TwoTowerDecoder(nn.Module):
    def __init__(self, hidden, dropout):
        super().__init__()

        def mlp():
            return nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden))

        self.artist, self.exhibition = mlp(), mlp()
        self.bias = nn.Linear(hidden, 1)

    def forward(self, h_artist, h_exhibition):
        return self.exhibition(h_exhibition) @ self.artist(h_artist).T + self.bias(h_artist).T


class LinkModel(nn.Module):
    def __init__(self, name, in_dims, hidden, layers, dropout, include_curators=True,
                 num_ids=None):
        super().__init__()
        if name in HETERO_MODELS:
            self.encoder = HETERO_MODELS[name](in_dims, edge_types(include_curators), hidden,
                                               layers, dropout, num_ids)
        elif name in HOMOGENEOUS_MODELS:
            self.encoder = HomogeneousEncoder(in_dims, hidden, layers, dropout, conv=name)
        elif name == "mlp":
            self.encoder = FeatureEncoder(in_dims, hidden, dropout)
        else:
            raise ValueError(name)
        self.decoder = TwoTowerDecoder(hidden, dropout)

    def forward(self, g, targets, candidates):
        """Scores [len(targets), len(candidates)]."""
        h_artist, h_exhibition = self.encoder(g)
        return self.decoder(h_artist[candidates], h_exhibition[targets])


def make_model(mcfg, graph, num_curators):
    in_dims = {t: graph[t].x.shape[1] for t in graph.node_types}
    has_curators = CURATOR in in_dims
    num_ids = {CURATOR: num_curators} if mcfg["curator_ids"] and has_curators else None
    return LinkModel(mcfg["name"], in_dims, mcfg["hidden"], mcfg["layers"], mcfg["dropout"],
                     include_curators=has_curators, num_ids=num_ids).to(DEVICE)

## Training

Each epoch takes one optimizer step per rolling training snapshot. The loss is BPR (or BCE) with
uniformly sampled negatives, and each target exhibition gets equal weight, as in the
per-exhibition evaluation. Model selection uses validation MAP@20 on curated exhibitions.

In [13]:
@dataclass
class Prepared:
    graph: object
    targets: torch.Tensor
    candidates: torch.Tensor
    labels: torch.Tensor        # bool [targets, candidates]


class GraphBuilder:
    def __init__(self, feats, model_cfg, seed=0, leak=False):
        self.feats = feats
        self.name = model_cfg["name"]
        self.include_curators = model_cfg["curator_mode"] != "none"
        self.max_group = model_cfg.get("max_group")
        self.seed = seed
        self.leak = leak  # sanity check only: target labels visible to message passing

    def __call__(self, snap):
        if self.name in HOMOGENEOUS_MODELS:
            g = build_homogeneous(snap, self.feats, self.max_group, self.seed, leak=self.leak)
        else:
            g = build_hetero(snap, self.feats, self.include_curators, leak=self.leak)
        return g.to(DEVICE)

    def prepare(self, snap):
        return Prepared(self(snap), torch.from_numpy(snap.targets).to(DEVICE),
                        torch.from_numpy(snap.candidates).to(DEVICE),
                        torch.from_numpy(snap.label_matrix()).to(DEVICE))


class GNNScorer:
    """Adapts a trained LinkModel to the shared evaluation protocol."""

    def __init__(self, model, builder, name):
        self.model, self.builder, self.name = model, builder, name

    @torch.no_grad()
    def score(self, snap):
        self.model.eval()
        scores = self.model(self.builder(snap), torch.from_numpy(snap.targets).to(DEVICE),
                            torch.from_numpy(snap.candidates).to(DEVICE))
        return scores.cpu().numpy()


def ranking_loss(scores, labels, kind, num_neg, gen):
    t_idx, c_idx = labels.nonzero(as_tuple=True)
    if len(t_idx) == 0:
        return None
    neg = torch.randint(0, labels.shape[1], (len(t_idx), num_neg), generator=gen).to(labels.device)
    valid = (~labels[t_idx[:, None], neg]).float()
    pos_scores = scores[t_idx, c_idx]
    neg_scores = scores[t_idx[:, None], neg]
    if kind == "bpr":
        per_pair = -(F.logsigmoid(pos_scores[:, None] - neg_scores) * valid).sum(1)
    elif kind == "bce":
        per_pair = (F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores),
                                                       reduction="none")[:, None]
                    + F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores),
                                                         reduction="none") * valid).sum(1)
    else:
        raise ValueError(kind)
    per_pair = per_pair / valid.sum(1).clamp(min=1)
    n_pos = labels.sum(1).float()
    return (per_pair / n_pos[t_idx]).sum() / (n_pos > 0).sum()


def train_model(data, feats, cutoff, cfg, seed, leak=False, verbose=True):
    """Train on snapshots before the validation window; select by validation MAP@K."""
    mcfg, ks = cfg["model"], cfg["eval"]["ks"]
    metric = cfg["eval"]["primary_metric"]
    torch.manual_seed(seed)
    np.random.seed(seed)
    gen = torch.Generator().manual_seed(seed)

    builder = GraphBuilder(feats, mcfg, seed, leak=leak)
    train = [builder.prepare(s) for s in train_snapshots(data, cutoff, cfg["split"])]
    val_snap, _ = eval_snapshots(data, cutoff, cfg["split"])

    model = make_model(mcfg, train[0].graph, data.num_curators)
    opt = torch.optim.Adam(model.parameters(), lr=mcfg["lr"], weight_decay=mcfg["weight_decay"])
    scorer = GNNScorer(model, builder, mcfg["name"])

    best = {"score": -np.inf, "epoch": 0, "state": None}
    history, stale = [], 0
    for epoch in range(1, mcfg["epochs"] + 1):
        model.train()
        losses = []
        for i in torch.randperm(len(train), generator=gen).tolist():
            p = train[i]
            loss = ranking_loss(model(p.graph, p.targets, p.candidates), p.labels,
                                mcfg["loss"], mcfg["num_neg"], gen)
            if loss is None:
                continue
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        if epoch % mcfg["eval_every"] == 0:
            per_exh, _ = evaluate(scorer, val_snap, ks, seed=seed)
            val = summarize(per_exh)["curated"][metric]
            history.append({"epoch": epoch, "loss": float(np.mean(losses)), f"val_{metric}": val})
            if verbose:
                print(f"  epoch {epoch:4d}  loss {np.mean(losses):.4f}  val {metric} {val:.4f}")
            if val > best["score"]:
                best = {"score": val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
                stale = 0
            else:
                stale += 1
                if stale >= mcfg["patience"]:
                    break

    model.load_state_dict(best["state"])
    return model, builder, {"best_epoch": best["epoch"], f"best_val_{metric}": best["score"],
                            "history": history}


SKIP_EXISTING = True   # reuse saved runs with the same config, e.g. after a Colab disconnect


def run_experiment(model_name, curator_mode, cutoff, seed, cfg=CFG, verbose=False):
    """Train one model, evaluate it on the test window (returning and debuting artists) and
    save per-exhibition results, positive ranks and weights."""
    cfg = copy.deepcopy(cfg)
    cfg["model"].update(name=model_name, curator_mode=curator_mode)
    tag = f"{model_name}_{curator_mode}"
    run = f"{tag}_T{cutoff}_s{seed}"
    out_dir = RESULTS_DIR / "gnn"
    out_dir.mkdir(exist_ok=True)
    info_file = out_dir / f"{run}_info.json"

    if SKIP_EXISTING and info_file.exists():
        saved = json.loads(info_file.read_text())
        if saved["model"] == cfg["model"] and saved.get("split") == cfg["split"]:
            summary, info = saved["summary"], saved
            print(f"{run:28s} loaded from {info_file.name}")
            return [{"model": tag, "cutoff": cutoff, "seed": seed, "subset": s,
                     "best_epoch": info["best_epoch"], **m} for s, m in summary.items()]

    start = time.time()
    run_data = data.with_shuffled_curators(seed) if curator_mode == "shuffled" else data
    model, builder, info = train_model(run_data, feats, cutoff, cfg, seed, verbose=verbose)
    _, test = eval_snapshots(run_data, cutoff, cfg["split"])
    scorer = GNNScorer(model, builder, tag)
    summary = {}
    for prefix, snap in [("", test), ("cold_", cold_start_snapshot(run_data, test))]:
        per_exh, ranks = evaluate(scorer, snap, cfg["eval"]["ks"], seed=seed)
        per_exh.to_csv(out_dir / f"{run}_{prefix}per_exh.csv", index=False)
        ranks.to_csv(out_dir / f"{run}_{prefix}ranks.csv", index=False)
        summary.update({prefix + s: m for s, m in summarize(per_exh).items()})
    torch.save(model.state_dict(), out_dir / f"{run}_model.pt")
    with open(info_file, "w") as f:
        json.dump({"model": cfg["model"], "split": cfg["split"], "summary": summary, **info},
                  f, indent=2, default=float)
    print(f"{run:28s} best epoch {info['best_epoch']:3d}  "
          f"test map@20 {summary['curated']['map@20']:.4f}  "
          f"recall@50 {summary['curated']['recall@50']:.4f}  ({time.time() - start:.0f}s)")
    return [{"model": tag, "cutoff": cutoff, "seed": seed, "subset": s,
             "best_epoch": info["best_epoch"], **m} for s, m in summary.items()]

## Sanity check: label leakage

This trains a second model with target labels visible to message passing, during both training
and testing. It should score far above the masked model: on the 1960 cutoff it reaches AUC
≈ 0.99 and MAP@20 ≈ 0.24, against ≈ 0.93 and ≈ 0.09 masked. The masked model's lower scores
therefore come from the masking, not from a model that cannot use the graph.

In [14]:
RUN_LEAK_CHECK = True
if RUN_LEAK_CHECK:
    leak_model, leak_builder, _ = train_model(data, feats, 1960, CFG, seed=0, leak=True,
                                              verbose=False)
    leaky, _ = evaluate(GNNScorer(leak_model, leak_builder, "leak"), test_1960, CFG["eval"]["ks"])
    print({k: round(v, 3) for k, v in summarize(leaky)["curated"].items()})

{'n_exhibitions': 163, 'recall@10': 0.31, 'map@10': 0.205, 'ndcg@10': 0.296, 'recall@20': 0.443, 'map@20': 0.204, 'ndcg@20': 0.323, 'recall@50': 0.623, 'map@50': 0.206, 'ndcg@50': 0.362, 'auc': 0.991, 'ap': 0.242}


## Design check: depth and curator identities (validation only)

Two choices were checked on the **validation** windows of the 1960 and 1970 cutoffs, with
2 seeds each, before any test results were computed:

- **2 vs 3 layers.** The path *target exhibition → curator → the curator's past exhibitions →
  their artists* is three hops long.
- **`curator_ids`.** Curator features describe a curator's demographics and activity, not who
  they are. Learned per-curator embeddings let the model recognize individual curators.

Set `RUN_TUNING = True` to repeat it (24 training runs). Otherwise the saved table is shown.
It was run before the collection features were added, so it compares architectures only.

**Outcome.** 3 layers beat 2 both with curators (0.122 vs 0.120 mean validation MAP@20) and
without (0.126 vs 0.114), so `layers = 3`. Curator ids added 0.001, which is within seed noise.
They stay on (`curator_ids = True`) so that the curator models get their best chance to use
curator identity, which makes the comparison with the `none` models conservative.

In [15]:
RUN_TUNING = False
VARIANTS = {"full, 2 layers": ("full", {"layers": 2}),
            "full, 3 layers": ("full", {"layers": 3}),
            "full, 2 layers + curator ids": ("full", {"layers": 2, "curator_ids": True}),
            "full, 3 layers + curator ids": ("full", {"layers": 3, "curator_ids": True}),
            "none, 2 layers": ("none", {"layers": 2}),
            "none, 3 layers": ("none", {"layers": 3})}
tuning_file = RESULTS_DIR / "gnn" / "tuning.csv"
if RUN_TUNING:
    rows = []
    for variant, (mode, overrides) in VARIANTS.items():
        for cutoff in [1960, 1970]:
            for seed in [0, 1]:
                cfg = copy.deepcopy(CFG)
                cfg["model"].update(name="rgcn", curator_mode=mode, **overrides)
                _, _, info = train_model(data, feats, cutoff, cfg, seed, verbose=False)
                rows.append({"variant": variant, "cutoff": cutoff, "seed": seed,
                             "best_epoch": info["best_epoch"],
                             "val_map@20": info["best_val_map@20"]})
    pd.DataFrame(rows).to_csv(tuning_file, index=False)
tuning = pd.read_csv(tuning_file)
tuning.pivot_table(index="variant", columns="cutoff", values="val_map@20").assign(
    mean=lambda t: t.mean(axis=1)).sort_values("mean", ascending=False).round(4)

cutoff,1960,1970,mean
variant,,,
"none, 3 layers",0.1493,0.1020,0.1256
"full, 3 layers + curator ids",0.1414,0.1046,0.1230
"full, 3 layers",0.1389,0.1055,0.1222
"full, 2 layers + curator ids",0.1447,0.0989,0.1218
"full, 2 layers",0.1403,0.0988,0.1196
"none, 2 layers",0.1328,0.0958,0.1143


## Experiments

- **Main ablation**: R-GCN with `full`, `none` and `shuffled` curators.
- **Second architecture**: HeteroConv with and without curators, plus HGT.
- **Exhibition-node ablation**: homogeneous `sage` / `gat` (artist co-exhibition graph) vs
  `rgcn_none`.
- **Graph ablation**: the feature-only `mlp` vs everything else.

Every run evaluates both the main task and the cold-start task. Runs that already exist in
`results/gnn/` with the same config are loaded, not retrained, so the grid can be resumed after a
Colab disconnect, or split between teammates sharing a Drive folder.

In [16]:
# (model, curator mode): seeds. The R-GCN ablation gets all 5 seeds, the rest 3.
EXPERIMENTS = {("rgcn", "full"): CFG["seeds"], ("rgcn", "none"): CFG["seeds"],
               ("rgcn", "shuffled"): CFG["seeds"],
               ("heteroconv", "full"): CFG["seeds"][:3], ("heteroconv", "none"): CFG["seeds"][:3],
               ("hgt", "full"): CFG["seeds"][:3],
               ("sage", "none"): CFG["seeds"][:3], ("gat", "none"): CFG["seeds"][:3],
               ("mlp", "none"): CFG["seeds"][:3]}
CUTOFFS = CFG["split"]["cutoffs"]

rows = []
for (model_name, mode), seeds in EXPERIMENTS.items():
    for cutoff in CUTOFFS:
        for seed in seeds:
            rows += run_experiment(model_name, mode, cutoff, seed)
gnn_table = pd.DataFrame(rows)
gnn_table.to_csv(RESULTS_DIR / "gnn" / "summary.csv", index=False)

rgcn_full_T1950_s0           loaded from rgcn_full_T1950_s0_info.json
rgcn_full_T1950_s1           loaded from rgcn_full_T1950_s1_info.json
rgcn_full_T1950_s2           loaded from rgcn_full_T1950_s2_info.json
rgcn_full_T1950_s3           loaded from rgcn_full_T1950_s3_info.json
rgcn_full_T1950_s4           loaded from rgcn_full_T1950_s4_info.json
rgcn_full_T1960_s0           loaded from rgcn_full_T1960_s0_info.json
rgcn_full_T1960_s1           loaded from rgcn_full_T1960_s1_info.json
rgcn_full_T1960_s2           loaded from rgcn_full_T1960_s2_info.json
rgcn_full_T1960_s3           loaded from rgcn_full_T1960_s3_info.json
rgcn_full_T1960_s4           loaded from rgcn_full_T1960_s4_info.json
rgcn_full_T1970_s0           loaded from rgcn_full_T1970_s0_info.json
rgcn_full_T1970_s1           loaded from rgcn_full_T1970_s1_info.json
rgcn_full_T1970_s2           loaded from rgcn_full_T1970_s2_info.json
rgcn_full_T1970_s3           loaded from rgcn_full_T1970_s3_info.json
rgcn_full_T1970_s4  

## Results

In [17]:
cols = ["recall@10", "recall@50", "map@20", "ndcg@20", "auc"]
curated = gnn_table[gnn_table["subset"] == "curated"]
baseline_file = RESULTS_DIR / "baselines" / "summary.csv"
if baseline_file.exists():
    base = pd.read_csv(baseline_file)
    curated = pd.concat([base[base["subset"] == "curated"], curated])
curated.pivot_table(index="model", columns="cutoff", values="map@20").round(4)

cutoff,1950,1960,1970,1980
model,,,,
adamic_adar,0.0832,0.0489,0.0406,0.0855
collection_popularity,0.0169,0.0623,0.0432,0.0417
common_neighbors,0.0854,0.0511,0.0464,0.0923
curator_collab,0.0969,0.0892,0.0667,0.1013
gat_none,0.0777,0.0820,0.0852,0.0977
heteroconv_full,0.0933,0.0958,0.0871,0.1114
heteroconv_none,0.0916,0.1026,0.0847,0.1003
hgt_full,0.0880,0.0904,0.0905,0.1137
mlp_none,0.0530,0.0734,0.0744,0.0875


In [18]:
# Mean (over seeds and cutoffs) on the curated subset.
curated.groupby("model")[cols].mean().sort_values("map@20", ascending=False).round(4)

,recall@10,recall@50,map@20,ndcg@20,auc
model,,,,,
rgcn_full,0.1310,0.3059,0.0990,0.1724,0.9315
heteroconv_full,0.1231,0.3035,0.0969,0.1683,0.9325
hgt_full,0.1281,0.3030,0.0956,0.1678,0.9334
heteroconv_none,0.1236,0.2946,0.0948,0.1672,0.9280
rgcn_none,0.1214,0.2898,0.0945,0.1652,0.9297
rgcn_shuffled,0.1185,0.2946,0.0917,0.1608,0.9272
curator_collab,0.1154,0.2919,0.0885,0.1560,0.8700
gat_none,0.1096,0.2708,0.0856,0.1508,0.9168
sage_none,0.1078,0.2651,0.0841,0.1479,0.9206


In [19]:
# Cold start (debuting artists only), curated subset. See 04_analysis for CIs.
cold = gnn_table[gnn_table["subset"] == "cold_curated"]
cold_file = RESULTS_DIR / "baselines" / "cold_summary.csv"
if cold_file.exists():
    cold = pd.concat([pd.read_csv(cold_file).query("subset == 'cold_curated'"), cold])
cold.groupby("model")[cols].mean().sort_values("map@20", ascending=False).round(4)

,recall@10,recall@50,map@20,ndcg@20,auc
model,,,,,
hgt_full,0.0196,0.0857,0.0107,0.0281,0.6480
rgcn_full,0.0199,0.0806,0.0106,0.0286,0.6439
heteroconv_full,0.0206,0.0761,0.0106,0.0269,0.6423
rgcn_shuffled,0.0188,0.0781,0.0099,0.0281,0.6445
rgcn_none,0.0225,0.0817,0.0098,0.0282,0.6537
sage_none,0.0179,0.0739,0.0091,0.0256,0.6403
mlp_none,0.0167,0.0735,0.0085,0.0246,0.6543
heteroconv_none,0.0151,0.0795,0.0081,0.0253,0.6515
gat_none,0.0119,0.0692,0.0071,0.0202,0.6491


### Curator ablation: paired bootstrap over test exhibitions

- **full − none** tests the overall value of curator information.
- **full − shuffled** tests whether the *specific* historical curator assignments matter beyond
  the extra connectivity.

In [20]:
for a, b in [("rgcn_full", "rgcn_none"), ("rgcn_full", "rgcn_shuffled"),
             ("rgcn_full", "curator_collab"), ("rgcn_none", "popularity")]:
    table = compare_runs(a, b, metric="map@20")
    if len(table):
        print(f"map@20: {a} − {b}")
        display(table.round(4))

map@20: rgcn_full − rgcn_none


,cutoff,seeds_a,seeds_b,rgcn_full,rgcn_none,mean_diff,ci_low,ci_high,n,p_le_0
0,1950,5,5,0.0901,0.0928,-0.0027,-0.0092,0.0040,131,0.785
1,1960,5,5,0.0927,0.0913,0.0014,-0.0053,0.0088,163,0.350
2,1970,5,5,0.0925,0.0868,0.0057,-0.0015,0.0125,248,0.067
3,1980,5,5,0.1209,0.1072,0.0137,0.0001,0.0266,206,0.025


map@20: rgcn_full − rgcn_shuffled


,cutoff,seeds_a,seeds_b,rgcn_full,rgcn_shuffled,mean_diff,ci_low,ci_high,n,p_le_0
0,1950,5,5,0.0901,0.0878,0.0023,-0.0088,0.0113,131,0.309
1,1960,5,5,0.0927,0.0931,-0.0003,-0.0071,0.0065,163,0.539
2,1970,5,5,0.0925,0.0841,0.0083,0.0032,0.0143,248,0.000
3,1980,5,5,0.1209,0.1017,0.0192,0.0089,0.0299,206,0.000


map@20: rgcn_full − curator_collab


,cutoff,seeds_a,seeds_b,rgcn_full,curator_collab,mean_diff,ci_low,ci_high,n,p_le_0
0,1950,5,1,0.0901,0.0969,-0.0068,-0.0312,0.0191,131,0.695
1,1960,5,1,0.0927,0.0892,0.0035,-0.0170,0.0221,163,0.363
2,1970,5,1,0.0925,0.0667,0.0257,0.0037,0.0487,248,0.008
3,1980,5,1,0.1209,0.1013,0.0196,-0.0041,0.0439,206,0.052


map@20: rgcn_none − popularity


,cutoff,seeds_a,seeds_b,rgcn_none,popularity,mean_diff,ci_low,ci_high,n,p_le_0
0,1950,5,1,0.0928,0.0749,0.0178,-0.0065,0.0427,131,0.074
1,1960,5,1,0.0913,0.0715,0.0198,0.0050,0.0349,163,0.005
2,1970,5,1,0.0868,0.0691,0.0177,0.0050,0.0304,248,0.002
3,1980,5,1,0.1072,0.0738,0.0334,0.0172,0.0495,206,0.000


## Test-time curator removal

The `none` ablation trains a different model. This instead takes one trained R-GCN, deletes the
curator edges of a single test exhibition, and re-scores that exhibition with the same weights.
The rank change of its true artists shows how much this model's predictions lean on those edges.
The output is used for the case studies in `04_analysis`.

In [21]:
def load_trained(model_name, curator_mode, cutoff, seed, cfg=CFG):
    """Rebuild a model saved by run_experiment, with its test snapshot."""
    cfg = copy.deepcopy(cfg)
    cfg["model"].update(name=model_name, curator_mode=curator_mode)
    run_data = data.with_shuffled_curators(seed) if curator_mode == "shuffled" else data
    builder = GraphBuilder(feats, cfg["model"], seed)
    _, test = eval_snapshots(run_data, cutoff, cfg["split"])
    model = make_model(cfg["model"], builder(test), data.num_curators)
    path = RESULTS_DIR / "gnn" / f"{model_name}_{curator_mode}_T{cutoff}_s{seed}_model.pt"
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.eval(), builder, test


def positive_ranks(scores, labels):
    """1-based rank of each positive among all candidates (ties count against it)."""
    return (scores[None, :] >= scores[labels][:, None]).sum(axis=1)


def curator_removal(model, builder, snap):
    scorer = GNNScorer(model, builder, "removal")
    labels = snap.label_matrix()
    base = scorer.score(snap)
    rows = []
    for i, e in enumerate(snap.targets):
        keep = snap.target_curator[0] != e
        if keep.all() or not labels[i].any():
            continue
        without = scorer.score(replace(snap, target_curator=snap.target_curator[:, keep]))[i]
        rows.append(pd.DataFrame({
            "exh_id": e, "artist": snap.candidates[labels[i]],
            "rank_with": positive_ranks(base[i], labels[i]),
            "rank_without": positive_ranks(without, labels[i])}))
    return pd.concat(rows, ignore_index=True)


model, builder, test = load_trained("rgcn", "full", 1960, 0)
removal = curator_removal(model, builder, test)
removal.to_csv(RESULTS_DIR / "gnn" / "curator_removal_rgcn_T1960_s0.csv", index=False)
change = removal["rank_without"] - removal["rank_with"]   # > 0: the curator edges helped
print(f"{len(removal)} true pairs in {removal['exh_id'].nunique()} curated test exhibitions")
print(f"removing curator edges worsens {(change > 0).mean():.0%} of ranks and improves "
      f"{(change < 0).mean():.0%}; median change {change.median():+.0f} places")

2335 true pairs in 163 curated test exhibitions
removing curator edges worsens 49% of ranks and improves 44%; median change +0 places
